# Mini-Batch GD

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes

In [2]:
X, Y = load_diabetes(return_X_y=True)

In [3]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

### OLS

In [4]:
from sklearn.linear_model import LinearRegression
lr1 = LinearRegression()

In [5]:
lr1.fit(x_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [6]:
lr1.coef_

array([  -9.15865318, -205.45432163,  516.69374454,  340.61999905,
       -895.5520019 ,  561.22067904,  153.89310954,  126.73139688,
        861.12700152,   52.42112238])

In [7]:
lr1.intercept_

np.float64(151.88331005254167)

In [8]:
y1_pred = lr1.predict(x_test)

In [9]:
from sklearn.metrics import r2_score

In [10]:
r2_score(y_test, y1_pred)

0.439933866156897

### Making MBGD

In [21]:
import random
class MBGD:

    def __init__(self, learning_rate, epoch, batch_size):
        self.lr = learning_rate
        self.epoch = epoch
        self.batch_size = batch_size
        self.coef = None
        self.intercept = None

    def fit(self, x_train, y_train):
        self.intercept = 0
        self.coef = np.ones(x_train.shape[1])

        for i in range(self.epoch):
            for j in range(x_train.shape[0]//self.batch_size):
                
                indx = random.sample(range(x_train.shape[0]), self.batch_size) # generate a list
                y_hat = self.intercept + np.dot(x_train[indx], self.coef)
            
                intercept_der = -2 * np.mean(y_train[indx] - y_hat)
                self.intercept = self.intercept - self.lr * intercept_der

                coef_der = -2 * np.dot((y_train[indx]-y_hat), x_train[indx]) / self.batch_size
                self.coef = self.coef - self.lr * coef_der

        print(self.intercept, self.coef)

    def predict(self, x_test):
        return self.intercept + np.dot(x_test, self.coef)

In [58]:
lr2 = MBGD(0.1, 100, 20)

In [59]:
lr2.fit(x_train, y_train)

155.134651420099 [  55.86858635  -65.15914131  346.04039067  247.95754836   24.81396059
  -22.81225412 -168.22863056  128.33325085  318.26099984  129.95458453]


In [60]:
y2_pred = lr2.predict(x_test)

In [61]:
r2_score(y_test, y2_pred)

0.4297143338225684

### Using SGD Regressor

In [63]:
from sklearn.linear_model import SGDRegressor

In [64]:
lr3 = SGDRegressor(learning_rate='constant', eta0=0.1)

In [65]:
batch_size = 20
for i in range(100): # epoch size
    indx = random.sample(range(x_train.shape[0]), batch_size)
    lr3.partial_fit(x_train[indx], y_train[indx])

In [66]:
lr3.intercept_

array([168.85757473])

In [67]:
lr3.coef_

array([  55.45726747,  -27.55567882,  275.06315835,  192.92040658,
         34.75621282,    0.67422502, -148.08544254,  131.8222145 ,
        251.96477475,  129.90707878])

In [68]:
y3_pred = lr3.predict(x_test)

In [69]:
r2_score(y_test, y3_pred)

0.3586676641755866